<a href="https://colab.research.google.com/github/Nikitoniy/Waka_projects/blob/main/%D0%95%D0%B6%D0%B5%D0%BC%D0%B5%D1%81%D1%8F%D0%BD%D1%8B%D0%B9_%D0%BE%D1%82%D1%87%D0%B5%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Обработка и структурирование данных о продажах

**Описание проекта:**

Проект направлен на автоматизацию подготовки и структурирования данных о продажах товаров. <u>Исходные данные содержат: названия товарных позиций, количество проданных единиц и стоимость продаж.</u> При этом названия товаров представлены в неунифицированном формате: могут отличаться написанием, регистром символов и обозначением числовых значений, а также содержать дополнительную информацию, не относящуюся к анализируемой продукции.

<u>Для дальнейшего анализа необходимо выделить из названия отдельные характеристики товара: бренд, модель и вкус.</u> Ручная обработка большого количества записей требует значительных временных затрат, поэтому в рамках проекта был разработан скрипт, автоматизирующий основные этапы подготовки данных.


**Основные этапы обработки:**

1. **Стандартизация числовых обозначений.** Приведение различных форматов записи чисел к единому виду. Например, обозначения 20k, 20.000 и тд преобразуются в 20000, если они относятся к одному и тому же числовому значению модели.

2. **Нормализация названий.** Приведение текста к нижнему регистру, исправление распространённых ошибок и вариантов написания брендов и моделей для унификации данных и упрощения дальнейшей обработки.

3. **Выделение характеристик товара.** Автоматическое извлечение бренда, модели и вкуса из исходного названия с последующим сохранением этих характеристик в отдельных столбцах таблицы.

4. **Очистка данных от нерелевантных категорий.** Исключение товарных позиций, не относящихся к целевой категории продукции, включая энергетические напитки, жевательный табак, кальянный табак и другие подобные товары.

5. **Формирование структурированной таблицы.** Подготовка итогового набора данных с унифицированными названиями и выделенными характеристиками товаров, сохранением информации о количестве продаж и стоимости.

# Библиотеки

In [16]:
import pandas as pd
import re
from google.colab import files

# Загрузка таблицы

In [17]:
#таблица
df = pd.read_excel("data.xlsx")

df.columns = ['name', 'total', 'price', 'TT']

display(df.head(10))
display(df.info())

,name,total,price,TT
0,WAKA SOPRO PA 10000,NaN,0,Охотный ряд
1,Waka 10000 Арбуз 2%,2шт,3000,Охотный ряд
2,Waka salt Виноград Яблоко 5 mg,3,3000,Охотный ряд
3,табак жеват. waka,2,3000,Охотный ряд
4,Waka SoPro PA 10000 ВИШНЕВАЯ ШИПУЧКА 2%,3шт,3000,Охотный ряд
5,Waka ESCOBAR Ягодный микс,3шт,3000,Охотный ряд
6,Waka ESCOBAR Лимонный пирог,3шт,3000,Охотный ряд
7,Waka PRO 20000 EXTRA Киви Маракуйя Гуава,3шт,3000,Охотный ряд
8,Waka PRO 20000 EXTRA Лимонад,2,3000,Охотный ряд
9,Waka JUPITER 30000 Малина Арбуз,3,3000,Охотный ряд


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    18 non-null     object
 1   total   17 non-null     object
 2   price   18 non-null     int64 
 3   TT      18 non-null     object
dtypes: int64(1), object(3)
memory usage: 708.0+ bytes


None

# Обработка

## Работа с колонкой **total**

In [18]:
#нишний регистр, удаляю ш/т и обратно в число
df['total'] = (
               df['total'].astype('str')
               .str.lower()
               .replace('[шт]', '', regex=True)
               .pipe(pd.to_numeric, errors='coerce')
               .abs()
              )

#убираю null и 0
df = (
      df
      .dropna(subset=['name', 'total'])
      .query('total > 0')
     )

#целое число
df['total'] = df['total'].astype(int)

## Работа с колонкой **name**

### Работа с числовыми значениями

In [19]:
#нижний регистр
df['name'] = df['name'].str.lower()

#запятая на точку
df['name'] = df['name'].replace(',', '.', regex=True)

#лишние пробелы
df['name'] = (
              df['name']
              .replace(r'\s+', ' ', regex=True)
              .str
              .strip()
             )

#числа с k
df['name'] = (
              df['name']
              .replace(r'(\d+)k', r'\g<1>000', regex=True)
             )

#пробелы справа и слева от букв
df['name'] = (
              df['name']
              .replace(r'([^\d\s])(\d+)', r'\1 \2', regex=True)
              .replace(r'(\d+)([^\d\s])', r'\1 \2', regex=True)
             )

#убираю у точки пробелы
df['name'] = (
              df['name']
              .str.replace(r'(?<=\d)\s*\.\s*(?=\d)', '.', regex=True)
             )

df['name'] = (
              df['name']
              .replace(
              r'(\d+)(?:\.\s?|\s)000\b',
              r'\g<1>000', regex=True)
             )

### Удаление лишних продуктов

In [20]:
#стоп слова
stop_words = [
              'жт',
              'кс',
              'гр',
              'гр.',
              'табак кальян.',
              'табак кал.',
              'табак жеват.',
              'глицерин',
              'стики',
              'табак для кальяна',
              'бестабачная смесь',
              'бестабачная безникотиновая смесь',
              'mpk',
              'мпк',
              'мрк',
              'напиток'
             ]

#паттерн удаления
pattern_del = r'\b(?:' + '|'.join(map(re.escape, stop_words)) + r')(?!\w)'

#очистка лишних строк по паттерну
df = df[~df['name'].str.contains(pattern_del, regex=True)]

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16 entries, 1 to 17
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    16 non-null     object
 1   total   16 non-null     int64 
 2   price   16 non-null     int64 
 3   TT      16 non-null     object
dtypes: int64(2), object(2)
memory usage: 640.0+ bytes


### Работа с названиями брендов

#### Название самих брендов

In [22]:
#podonki
df['name'] = (
              df['name']
              .replace('подонки', 'podonki', regex=True)
             )

#iforce
df['name'] = (
              df['name']
              .replace('i:force', 'iforce', regex=True)
             )

#puffmi
df['name'] = (
              df['name']
              .replace(r'\buffmi\b', 'puffmi', regex=True)
             )

#duall
df['name'] = (
              df['name']
              .replace(r'\bdual\b', 'duall', regex=True)
             )

#fummo
df['name'] = (
              df['name']
              .replace([r'\bfumo\b', r'\bфумо\b', r'\bфуммо\b'],
                       'fummo', regex=True)
             )

#elf bar
df['name'] = (
              df['name']
              .replace(['elfbar', 'элфбар', 'элф бар',
                        'efi bar', 'efibar', 'efl bar', 'eflbar'],
                       'elf bar', regex=True)
             )

#narcoz
df['name'] = (
              df['name']
              .replace(['narсoz', 'narkoz', 'наркоз'],
                       'narcoz', regex=True)
             )

#tikobar
df['name'] = (
              df['name']
              .replace(['tokobar', 'tiko bar'],
                       'tikobar', regex=True)
             )

#inflave
df['name'] = (
              df['name']
              .replace(regex={
                  'loona elixir': 'inflave loona elixir',
                  r'\bnflave\b': 'inflave'
              })
             )

#bryzgi
df['name'] = (
              df['name']
              .replace([r'\bbz\b', r'\bbg\b'],
                       'bryzgi', regex=True)
             )

#geek bar
df['name'] = (
              df['name']
              .replace(['geekbar', 'гикбар', 'гик бар', r'\bgb\b'],
                       'geek bar', regex=True)
             )

#geek vape
df['name'] = (
              df['name']
              .replace(['geekvape', 'гиквейп', 'гик вейп', r'\bgv\b'],
                       'geek vape', regex=True)
             )

#lost mary
df['name'] = (
              df['name']
              .replace(['lostmary', 'lost mery', 'lostmery',
                        'lost marry', 'lostmarry',
                        'лост мери', 'лостмери',
                        'лост мари', 'лостмари'],
                       'lost mary', regex=True)
             )

#lm xlink
df['name'] = (
              df['name']
              .replace(['xlink', 'x-link', 'x link'],
                       'lost mary x-link', regex=True)
             )

#plonq
df['name'] = (
              df['name']
              .replace(['plong', 'плонк', 'плонг'],
                       'plonq', regex=True)
             )

#### Название моделей

In [23]:
df['name'] = (
              df['name']
              .replace(regex={'gang 20': 'gang arctic 20',
                              'gang 16': 'gang immortal 16',
                              'gang 25': 'gang armor 25',
                              'gang box': 'gang xbox',
                              'fummo - aqua': 'fummo aqua',
                              r'\broqy\b': 'plonq roqy',
                              r'\bmeta\b': 'plonq meta',
                              r'\bmaxs\b': 'max s',
                              r'\bmax smart\b': 'max s',
                              r'\bmaxsmart\b': 'max s',
                              'bm 15000': 'bm 16000',
                              'wenax': 'geek vape wenax',
                              'pasito': 'smoant pasito',
                              'charon': 'smoant charon',
                              'santi': 'smoant santi',
                              'eskobar': 'waka escobar',
                              'esсobar': 'waka escobar',
                              r'\bvmate\b': 'voopoo vmate',
                              r'\bv mate\b': 'voopoo vmate',
                              r'\bminican\b': 'brusko minican',
                              r'\bmini can\b': 'brusko minican',
                              r'\bxros\b': 'vaporesso xros',
                              r'\bx ros\b': 'vaporesso xros',
                              'lil solid': 'iqos lil solid'
                             })
              )

# Доработка <a id="revision"></a>

## Названия брендов

### Все названия брендов

In [24]:
#названия всех брендов
brand_pool = [
              'Монашка', 'Самоубийца', 'ANGRY VAPE', 'ARTERY', 'ATTACKER',
              'brusko', 'BRYZGI', 'BUBBLE MON', 'BUGATTI', 'CANDYMAN',
              'CATACLYSM', 'CNPT', 'COFFY', 'DUALL', 'ElectroJam', 'ELF BAR',
              'Elfliq', 'ELUX', 'FALCON', 'FIREROSE', 'Fizz', 'FLONQ', 'FLUM',
              'FUMARI', 'FUMMO', 'Funky Lands', 'GANG', 'Geek Bar', 'Geek Vape',
              'GLO', 'HOTSPOT', 'HQD', 'HUSKY', 'HYLA', 'HYLINET', 'ICEBERG',
              'iFORCE', 'IGNITE', 'INFLAVE', 'INSTABAR', 'IQOS', 'IZI',
              'LOST MARY', 'MOJI', 'MONSTERVAPOR', 'MOTI', 'OXBAR', 'PAFOS',
              'PLONQ', 'PODONKI', 'PUFFMI', 'РИК И МОРТИ', 'SKALA',
              'SKE', 'SMOANT', 'SOAK', 'SPIRIT', 'STARLINE', 'Terea', 'Tikobar',
              'TYSON', 'UDN', 'UTBAR', 'VARMILO', 'VIENTO', 'VLAGA', 'VLIQ',
              'VOOPOO', 'VOZOL', 'VUSE', 'WAKA', 'Yooz',
              'Vaporesso', 'LAISKA', 'OUKITEL', 'NANCY', 'CoolPlay', 'Chillax',
              'Trava', 'swonq', 'Catswill', 'MELOSO', 'Nasty', 'Анархия',
              'snoopysmoke', 'Dabbler', 'PLENA', 'Inspo', 'Catstrip',
              'Chrome', 'BJORN', 'ASPARK', 'NARCOZ', 'MiVio', 'MIKING',
              'Alaska', 'vers', 'chakra', 'oggo', 'qwerty', 'joystick', 'jellybox'
             ]

brand_pool = list(map(str.lower, brand_pool))

### Поиск бренда

In [25]:
#новая колонка с брендом
df['brand'] = df['name'].apply(
                               lambda x: ', '.join(
                                   brand for brand in brand_pool
                                     if isinstance(x, str) and re.search(
                                        rf'\b{re.escape(brand)}\b',
                                        x
                                      )
                                  )
                              )

## Названия моделей

### Поиск жидкостей, ароматизаторов, испарителей и картриджей

In [26]:
#бренд + жидкости и остальное
def find_product_type(name):

    if not isinstance(name, str):
        return ''

    name = name.lower()

    #ищу бренд
    brand = next(
                  (
                   b for b in brand_pool
                   if re.search(rf'\b{re.escape(b)}\b', name)
                  ),
                  ''
                )

    if re.search(r'\b(испаритель|испарители|coil)\b', name):
        product_type = 'испаритель'

    elif re.search(r'\bароматизатор\b', name):
        product_type = 'ароматизатор'

    elif re.search(r'\b(картр|катр)', name):
        product_type = 'картридж'

    elif re.search(
                   r'\b(ml|мл|mg|мг|ml\.|mg\.|мл\.|мг\.|salt)\b',
                   name
                  ):
        product_type = 'жидкость'

    else:
        product_type = ''

    if brand and product_type:
        return f'{brand} - {product_type}'

    return ''

df['model'] = df['name'].apply(find_product_type)

df.head(10)

,name,total,price,TT,brand,model
1,waka 10000 арбуз 2 %,2,3000,Охотный ряд,waka,
2,waka salt виноград яблоко 5 mg,3,3000,Охотный ряд,waka,waka - жидкость
4,waka sopro pa 10000 вишневая шипучка 2 %,3,3000,Охотный ряд,waka,
5,waka escobar ягодный микс,3,3000,Охотный ряд,waka,
6,waka escobar лимонный пирог,3,3000,Охотный ряд,waka,
7,waka pro 20000 extra киви маракуйя гуава,3,3000,Охотный ряд,waka,
8,waka pro 20000 extra лимонад,2,3000,Охотный ряд,waka,
9,waka jupiter 30000 малина арбуз,3,3000,Охотный ряд,waka,
10,waka jupiter 30000 мята,2,3000,Охотный ряд,waka,
11,waka 10000 вишня 2 %,1,3000,Охотный ряд,waka,


### Все дополнительные названия моделей

In [27]:
model_pool = [
              'ultra', 'extra', 'hard', 'plus', 'pro', 'max', 'prime',
              'meta', 'alpha', 'roqy', 'x-link', 'puffball', 'jupiter',
              'triplex', 'cuvie bar', 'cuvie', 'glaze', 'neo', 'escobar',
              'strong', 'minican', 'zero', 'pasito', 'charon'
             ]

### Поиск модели

In [28]:
#бренд + тяги + теги
def find_brand_model(name):

    if not isinstance(name, str):
        return ''

    brand = next(
                  (
                   brand for brand in brand_pool
                   if re.search(rf'\b{re.escape(brand)}\b', name)
                  ), ''
                )

    pulls = re.search(r'(?<!\d)(\d{3,})(?!\d)', name)

    if not brand:
        return ''

    if not pulls:
        pulls = ''

    else:
        pulls = pulls.group(1)

    model = next(
                (
                 model for model in model_pool
                 if re.search(rf'\b{re.escape(model)}\b', name)
                ), ''
              )

    if model:
        return f'{brand} {pulls} {model}'

    return f'{brand} {pulls}'

#маска
mask = df['model'] == ''
df.loc[mask, 'model'] = df.loc[mask, 'name'].apply(find_brand_model)

#лишние пробелы
df['model'] = (
              df['model']
              .replace(r'\s+', ' ', regex=True)
              .str
              .strip()
             )

### Названия вкусов

In [29]:
#ищу последние слова от 3 букв
def find_flavor(name):

    name = str(name)
    name = re.sub(r'[^а-яА-ЯёЁa-zA-Z0-9\s]', ' ', name)

    words = name.split()
    result = []

    for word in reversed(words):

        if re.fullmatch(r'[а-яА-ЯёЁ]{3,}', word):
            result.append(word)

        elif result and (
                         re.search(r'\d', word) or
                         re.fullmatch(r'[a-zA-Z]+', word)
                        ):
            break

    return ' '.join(reversed(result))

df['flavor'] = df['name'].apply(find_flavor)

# Выгрузка финал

In [30]:
#df.to_csv('monthly_rep.csv', index=False, encoding='utf-8-sig')
#files.download('monthly_repe.csv')

display(df.head(10))
display(df.info())

,name,total,price,TT,brand,model,flavor
1,waka 10000 арбуз 2 %,2,3000,Охотный ряд,waka,waka 10000,арбуз
2,waka salt виноград яблоко 5 mg,3,3000,Охотный ряд,waka,waka - жидкость,виноград яблоко
4,waka sopro pa 10000 вишневая шипучка 2 %,3,3000,Охотный ряд,waka,waka 10000,вишневая шипучка
5,waka escobar ягодный микс,3,3000,Охотный ряд,waka,waka escobar,ягодный микс
6,waka escobar лимонный пирог,3,3000,Охотный ряд,waka,waka escobar,лимонный пирог
7,waka pro 20000 extra киви маракуйя гуава,3,3000,Охотный ряд,waka,waka 20000 extra,киви маракуйя гуава
8,waka pro 20000 extra лимонад,2,3000,Охотный ряд,waka,waka 20000 extra,лимонад
9,waka jupiter 30000 малина арбуз,3,3000,Охотный ряд,waka,waka 30000 jupiter,малина арбуз
10,waka jupiter 30000 мята,2,3000,Охотный ряд,waka,waka 30000 jupiter,мята
11,waka 10000 вишня 2 %,1,3000,Охотный ряд,waka,waka 10000,вишня


<class 'pandas.core.frame.DataFrame'>
Index: 16 entries, 1 to 17
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    16 non-null     object
 1   total   16 non-null     int64 
 2   price   16 non-null     int64 
 3   TT      16 non-null     object
 4   brand   16 non-null     object
 5   model   16 non-null     object
 6   flavor  16 non-null     object
dtypes: int64(2), object(5)
memory usage: 1.0+ KB


None